<a href="https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My lane: Lane 2 — Refresh / Content Opportunity Scoring

I am choosing Lane 2: Refresh / Content Opportunity Scoring. The decision I want to support is which pages should be reviewed first when a content or SEO team has limited time. Rather than treating every page equally, I want to investigate whether safe content and search signals can help rank pages by priority. This lane fits the starter pipeline I have already explored, where a hand-written baseline and learned models were used to prioritize pages. I am choosing it provisionally because it provides a clear connection between the available data, a practical decision, and an action someone can take, while still allowing me to refine the exact target and approach as I learn more about the full warehouse data.

In [15]:
print("Selected lane: Lane 2 — Refresh / Content Opportunity Scoring")
print("Unit of analysis: one page")
print("Provisional choice: subject to refinement through Week 4")

Selected lane: Lane 2 — Refresh / Content Opportunity Scoring
Unit of analysis: one page
Provisional choice: subject to refinement through Week 4


## The question

**Research question:** Which pages should a content or SEO team review first for possible refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis:** One page.

**Decision:** The work aims to improve the decision of which pages should receive limited editorial or SEO review time first.

**Who acts and what do they do?** A content editor or SEO team would use the ranked output to investigate the highest-priority pages and decide on an appropriate action, such as refreshing content, expanding it, protecting a strong page, pruning or merging weak content, or monitoring the page.

**Output:** The intended output is a ranked action queue containing a priority score, supporting reason codes, and eventually an appropriate action suggestion or confidence label.

**Cost of a wrong recommendation:** A false positive could waste limited editor or analyst time reviewing a page that was not actually a high priority. A false negative could cause the team to overlook a genuinely important page or opportunity, potentially allowing further loss of visibility, clicks, or engagement before it is reviewed.

**Why data or ML may help:** Page priority may depend on several signals interacting, including visibility, position, CTR, freshness, engagement, and content characteristics. A simple rule may miss useful combinations of signals. However, ML will only be useful if it improves the decision beyond a transparent baseline; otherwise, a simpler rule or analysis may be preferable.

In [16]:
decision_frame = {
    "unit_of_analysis": "one page",
    "decision": "Which pages should be reviewed first?",
    "output": "Ranked action queue",
    "primary_metric": "Precision@K"
}

for key, value in decision_frame.items():
    print(f"{key}: {value}")

unit_of_analysis: one page
decision: Which pages should be reviewed first?
output: Ranked action queue
primary_metric: Precision@K


## Quick look at the data

The starter dataset contains **30,000 pages and 44 columns**. Of these pages, **16,262 (54.2%)** are marked as declining. This suggests that there can be many potential pages competing for limited review time, making prioritization a meaningful problem rather than simply reviewing every page equally.

There is also a difference in median visibility between pages in different trend groups: pages marked **down have a median of 961 impressions over 90 days**, compared with **587 for pages marked up**. This suggests that decline alone may not be enough to determine priority. A declining page with substantial existing visibility could potentially deserve more urgent review than a lower-visibility page.

Finally, the correlation between **search_volume and impressions_90d is 0.001**, which is effectively zero in this dataset. This observed result suggests that keyword search volume alone is not useful for predicting the actual visibility a page receives here. A prioritization approach may therefore need to consider multiple signals together rather than relying on one simple measure.

These are exploratory observations from the starter dataset. They make Lane 2 worth investigating, but they do not prove that a particular model or intervention will improve page performance.

In [17]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current working directory:
/content/flyrank-ml-internship

Files/folders here:
['work', 'outputs', 'README.md', '.github', '.gitignore', 'docs', 'data', 'SETUP.md', 'scripts', 'LICENSE', 'submission', 'DATA_USE.md', 'GUIDE.md', 'skills', 'requirements.txt', 'AGENTS.md', '.git', 'notebooks', 'CLAUDE.md']


In [18]:
!find /content -name "content_refresh_anonymized.csv" 2>/dev/null

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [19]:
!git clone https://github.com/mbinaqeel-analyst/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 39), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 12.95 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/flyrank-ml-internship


In [20]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


In [21]:
from pathlib import Path
import pandas as pd

# Find the repository root
repo_root = Path.cwd()
while not (repo_root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not find the starter dataset.")
    repo_root = repo_root.parent

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print(f"Dataset loaded successfully: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Path: {data_path}")

# 1. Declining pages
total_pages = len(df)
declining_pages = (df["trend_direction"] == "down").sum()
declining_pct = declining_pages / total_pages * 100

print(f"\nTotal pages: {total_pages:,}")
print(f"Declining pages: {declining_pages:,} ({declining_pct:.1f}%)")

# 2. Median impressions: declining vs growing pages
median_impressions = (
    df[df["trend_direction"].isin(["down", "up"])]
    .groupby("trend_direction")["impressions_90d"]
    .median()
)

print("\nMedian impressions_90d:")
print(median_impressions.round(1).to_string())

# 3. Search volume vs actual impressions
corr = df["search_volume"].corr(df["impressions_90d"])

print(f"\nCorrelation between search_volume and impressions_90d: {corr:.3f}")

Dataset loaded successfully: 30,000 rows, 44 columns
Path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Total pages: 30,000
Declining pages: 16,262 (54.2%)

Median impressions_90d:
trend_direction
down    961.0
up      587.0

Correlation between search_volume and impressions_90d: 0.001


## Careful words: what I can and can't claim

This work can describe observed patterns in the available data, measure directional relationships, compare models or rules on a defined validation strategy, and provide decision support for prioritizing pages for human review.

The final ranked queue would not prove that a recommended action will improve a page. For example, identifying a declining page does not guarantee that refreshing it will increase traffic or visibility. Likewise, an observed relationship between a feature and an outcome does not establish that the feature caused the outcome.

I will not claim to have discovered, proved, or predicted Google's algorithm. The data is observational, so my conclusions will be limited to what was measured in the available dataset and validation setup. Any ML output should be treated as decision support rather than an automatic instruction. Final actions should remain subject to human review and the limitations of the data and model.

In [22]:
claims = [
    "Observed patterns in the available data",
    "Directional relationships",
    "Decision support for prioritization",
    "Comparison against a defined baseline and validation strategy"
]

not_claims = [
    "Causal proof",
    "Proof of a Google algorithm factor",
    "A guarantee that refreshing a page will improve performance"
]

print("This work CAN claim:")
for claim in claims:
    print("-", claim)

print("\nThis work CANNOT claim:")
for claim in not_claims:
    print("-", claim)

This work CAN claim:
- Observed patterns in the available data
- Directional relationships
- Decision support for prioritization
- Comparison against a defined baseline and validation strategy

This work CANNOT claim:
- Causal proof
- Proof of a Google algorithm factor
- A guarantee that refreshing a page will improve performance


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.